In [10]:
import string
from pprint import pprint

class Vectorizer:
    def standardize(self, text):
        text = text.lower()
        return "".join(char for char in text if char not in string.punctuation)
    
    def tokenize(self, text):
        text = self.standardize(text)
        return text.split()

    def make_vocabulary(self, dataset):
        self.vocabulary = {"": 0, "[UNK]": 1}
        for text in dataset:
            text = self.standardize(text)
            tokens = self.tokenize(text)
            for token in tokens:
                if token not in self.vocabulary:
                    self.vocabulary[token] = len(self.vocabulary)
        self.inverse_vocabulary = dict((v, k) for k, v in self.vocabulary.items())

    def encode(self, text):
        text = self.standardize(text)
        tokens = self.tokenize(text)
        return [self.vocabulary.get(token, 1) for token in tokens]
    
    def decode(self, int_sequence):
        return " ".join(self.inverse_vocabulary.get(i, "[UNK]") for i in int_sequence)
    
vectorizer = Vectorizer()
dataset = [           
    "I write, erase, rewrite",   
    "Erase again, and then",
    "A poppy blooms.",
]
vectorizer.make_vocabulary(dataset)
pprint(vectorizer.vocabulary)

{'': 0,
 '[UNK]': 1,
 'a': 9,
 'again': 6,
 'and': 7,
 'blooms': 11,
 'erase': 4,
 'i': 2,
 'poppy': 10,
 'rewrite': 5,
 'then': 8,
 'write': 3}


In [11]:
test_sentence = "I write, rewrite, and still rewrite again"
encoded_sentence = vectorizer.encode(test_sentence)
print(encoded_sentence)

[2, 3, 5, 7, 1, 5, 6]


In [12]:
decoded_sentence = vectorizer.decode(encoded_sentence)
print(decoded_sentence)

i write rewrite and [UNK] rewrite again


In [15]:
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed
100 80.22M 100 80.22M   0      0  3.24M      0   00:24   00:24          5.44M


In [16]:
!rm -r aclImdb/train/unsup

In [26]:
!cat aclImdb/train/pos/4077_10.txt

I first saw this back in the early 90s on UK TV, i did like it then but i missed the chance to tape it, many years passed but the film always stuck with me and i lost hope of seeing it TV again, the main thing that stuck with me was the end, the hole castle part really touched me, its easy to watch, has a great story, great music, the list goes on and on, its OK me saying how good it is but everyone will take there own best bits away with them once they have seen it, yes the animation is top notch and beautiful to watch, it does show its age in a very few parts but that has now become part of it beauty, i am so glad it has came out on DVD as it is one of my top 10 films of all time. Buy it or rent it just see it, best viewing is at night alone with drink and food in reach so you don't have to stop the film.<br /><br />Enjoy

In [28]:
import os
import pandas as pd

def text_dataset_from_directory(directory_path):
    texts = []
    labels = []
    
    # Each subdirectory is treated as a class
    for label in os.listdir(directory_path):
        class_dir = os.path.join(directory_path, label)
        if os.path.isdir(class_dir):
            for fname in os.listdir(class_dir):
                file_path = os.path.join(class_dir, fname)
                if os.path.isfile(file_path):
                    with open(file_path, encoding="utf-8") as f:
                        texts.append(f.read())
                        labels.append(label)
    
    # Return as a DataFrame for convenience
    return pd.DataFrame({"text": texts, "label": labels})

# Example usage:
dataset = text_dataset_from_directory("aclImdb/train")
print(dataset.head())
print(dataset.tail())

                                                text label
0  I just saw this film last night in the 2006 Tr...   neg
1  I watched this immediately after seeing HILLSI...   neg
2  I love Jane Austen's stories. I've only read t...   neg
3  This was disappointing. It started well enough...   neg
4  I question the motive of the creators of this ...   neg
                                                    text label
24995  This is a very beautiful and almost meditative...   pos
24996  All this talk about this being a bad movie is ...   pos
24997  ***SPOILERS*** ***SPOILERS***<br /><br />This ...   pos
24998  Paulie is a fantasy of a littler girl or perha...   pos
24999  This movie was made in Hungary i think. anyway...   pos


In [32]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# Step 1: Load text dataset from directory
def text_dataset_from_directory(directory_path):
    texts, labels = [], []
    for label in os.listdir(directory_path):
        class_dir = os.path.join(directory_path, label)
        if os.path.isdir(class_dir):
            for fname in os.listdir(class_dir):
                file_path = os.path.join(class_dir, fname)
                if os.path.isfile(file_path):
                    with open(file_path, encoding="utf-8") as f:
                        texts.append(f.read())
                        labels.append(label)
    return pd.DataFrame({"text": texts, "label": labels})

# Step 2: Prepare dataset
dataset = text_dataset_from_directory("aclImdb/train")
X_train, X_val, y_train, y_val = train_test_split(
    dataset["text"], dataset["label"], test_size=0.2, stratify=dataset["label"], random_state=42
)
print(f"Training samples: {len(X_train)}, Validation samples: {len(X_val)}")

Training samples: 20000, Validation samples: 5000


In [56]:
import re
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.base import BaseEstimator, TransformerMixin

class UnigramTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, max_features=None):
        self.max_features = max_features
        self.vocab_ = {}
        self.feature_names_ = []

    def _tokenize(self, text):
        return re.findall(r"\b\w+\b", text.lower())

    def fit(self, X, y=None):
        freq = {}
        for doc in X:
            for token in set(self._tokenize(doc)):  # use set to avoid double-counting
                freq[token] = freq.get(token, 0) + 1

        sorted_tokens = sorted(freq.items(), key=lambda x: x[1], reverse=True)
        if self.max_features:
            sorted_tokens = sorted_tokens[:self.max_features]

        self.vocab_ = {token: idx for idx, (token, _) in enumerate(sorted_tokens)}
        self.feature_names_ = list(self.vocab_.keys())
        return self

    def transform(self, X):
        rows, cols, data = [], [], []
        for row_idx, doc in enumerate(X):
            tokens = set(self._tokenize(doc))  # set ensures binary presence
            for token in tokens:
                if token in self.vocab_:
                    rows.append(row_idx)
                    cols.append(self.vocab_[token])
                    data.append(1)  # always 1, not counts
        return csr_matrix((data, (rows, cols)), shape=(len(X), len(self.vocab_)))

    def get_feature_names_out(self):
        return np.array(self.feature_names_)

docs = ["The cat sat on the mat.", "The dog chased the cat."]
uni_bin = UnigramTransformer(max_features=10)
X_bin = uni_bin.fit_transform(docs)

print("Vocabulary:", uni_bin.get_feature_names_out())
print("Matrix:\n", X_bin.toarray())

Vocabulary: ['the' 'cat' 'sat' 'on' 'mat' 'dog' 'chased']
Matrix:
 [[1 1 1 1 1 0 0]
 [1 1 0 0 0 1 1]]


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

pipe = Pipeline([
    ("unigrams", UnigramTransformer(max_features=5000)),
    ("rf", RandomForestClassifier(n_estimators=200, random_state=42))
])

pipe.fit(X_train, y_train)
print(pipe.score(X_val, y_val))

0.843


In [57]:
import re
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.base import BaseEstimator, TransformerMixin

class BigramTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, max_features=1000):
        self.max_features = max_features
        self.vocab_ = {}
        self.feature_names_ = []

    def _tokenize(self, text):
        return re.findall(r"\b\w+\b", text.lower())

    def fit(self, X, y=None):
        freq = {}

        # Count unigrams and bigrams
        for doc in X:
            tokens = self._tokenize(doc)
            # Unigrams
            for tok in tokens:
                freq[tok] = freq.get(tok, 0) + 1
            # Bigrams
            for i in range(len(tokens)-1):
                bg = f"{tokens[i]}_{tokens[i+1]}"
                freq[bg] = freq.get(bg, 0) + 1

        # Sort by frequency (descending)
        sorted_tokens = sorted(freq.items(), key=lambda x: x[1], reverse=True)

        # Keep top-n
        top_tokens = sorted_tokens[:self.max_features]

        # Build vocab
        self.vocab_ = {tok: idx for idx, (tok, _) in enumerate(top_tokens)}
        self.feature_names_ = [tok for tok, _ in top_tokens]
        return self

    def transform(self, X):
        rows, cols, data = [], [], []
        for row_idx, doc in enumerate(X):
            tokens = self._tokenize(doc)
            unigrams = set(tokens)
            bigrams = set([f"{tokens[i]}_{tokens[i+1]}" for i in range(len(tokens)-1)])
            present = unigrams.union(bigrams)
            for tok in present:
                if tok in self.vocab_:
                    rows.append(row_idx)
                    cols.append(self.vocab_[tok])
                    data.append(1)  # binary presence
        return csr_matrix((data, (rows, cols)), shape=(len(X), len(self.vocab_)))

    def get_feature_names_out(self):
        return np.array(self.feature_names_)

docs = [
    "The cat sat on the mat.",
    "The dog chased the cat.",
    "Dogs and cats are friends.",
    "The book is on the table.",
    "The cat sat on the book.",
]

bigram = BigramTransformer(max_features=10)
X_top = bigram.fit_transform(docs)

print("Selected features:", bigram.get_feature_names_out())
print("Matrix:\n", X_top.toarray())

Selected features: ['the' 'cat' 'on' 'the_cat' 'on_the' 'sat' 'cat_sat' 'sat_on' 'book'
 'the_book']
Matrix:
 [[1 1 1 1 1 1 1 1 0 0]
 [1 1 0 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0]
 [1 0 1 0 1 0 0 0 1 1]
 [1 1 1 1 1 1 1 1 1 1]]


In [58]:
pipe = Pipeline([
    ("bigrams", BigramTransformer(max_features=5000)),
    ("rf", RandomForestClassifier(n_estimators=200, random_state=42))
])

pipe.fit(X_train, y_train)
print(pipe.score(X_val, y_val))

0.8478


In [59]:
import re
import numpy as np
from math import log
from scipy.sparse import csr_matrix
from sklearn.base import BaseEstimator, TransformerMixin

class TfidfTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, max_features=None):
        self.max_features = max_features
        self.vocab_ = {}
        self.idf_ = {}
        self.feature_names_ = []

    def _tokenize(self, text):
        return re.findall(r"\b\w+\b", text.lower())

    def fit(self, X, y=None):
        # Count document frequencies
        df = {}
        total_docs = len(X)

        for doc in X:
            tokens = set(self._tokenize(doc))
            for tok in tokens:
                df[tok] = df.get(tok, 0) + 1

        # Sort by frequency
        sorted_tokens = sorted(df.items(), key=lambda x: x[1], reverse=True)

        # Limit features
        if self.max_features:
            sorted_tokens = sorted_tokens[:self.max_features]

        # Build vocab
        self.vocab_ = {tok: idx for idx, (tok, _) in enumerate(sorted_tokens)}
        self.feature_names_ = list(self.vocab_.keys())

        # Compute IDF
        self.idf_ = {
            tok: log(total_docs / (1 + df[tok])) for tok in self.vocab_
        }
        return self

    def transform(self, X):
        rows, cols, data = [], [], []
        for row_idx, doc in enumerate(X):
            tokens = self._tokenize(doc)
            total_terms = len(tokens)
            tf_counts = {}
            for tok in tokens:
                if tok in self.vocab_:
                    tf_counts[tok] = tf_counts.get(tok, 0) + 1
            for tok, count in tf_counts.items():
                col_idx = self.vocab_[tok]
                tf = count / total_terms
                val = tf * self.idf_[tok]
                rows.append(row_idx)
                cols.append(col_idx)
                data.append(val)
        return csr_matrix((data, (rows, cols)), shape=(len(X), len(self.vocab_)))

    def get_feature_names_out(self):
        return np.array(self.feature_names_)


docs = [
    "The cat sat on the mat.",
    "The dog chased the cat.",
    "Dogs and cats are friends.",
    "The book is on the table.",
    "The cat sat on the book.",
]

tfidf = TfidfTransformer(max_features=20)
X_tfidf = tfidf.fit_transform(docs)

print("Vocabulary:", tfidf.get_feature_names_out())
print("TF-IDF matrix:\n", X_tfidf.toarray())

Vocabulary: ['the' 'cat' 'on' 'sat' 'book' 'mat' 'dog' 'chased' 'are' 'and' 'friends'
 'dogs' 'cats' 'is' 'table']
TF-IDF matrix:
 [[0.         0.03719059 0.03719059 0.0851376  0.         0.15271512
  0.         0.         0.         0.         0.         0.
  0.         0.         0.        ]
 [0.         0.04462871 0.         0.         0.         0.
  0.18325815 0.18325815 0.         0.         0.         0.
  0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.18325815 0.18325815 0.18325815 0.18325815
  0.18325815 0.         0.        ]
 [0.         0.         0.03719059 0.         0.0851376  0.
  0.         0.         0.         0.         0.         0.
  0.         0.15271512 0.15271512]
 [0.         0.03719059 0.03719059 0.0851376  0.0851376  0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.        ]]


In [60]:
pipe = Pipeline([
    ("tfidf", TfidfTransformer(max_features=5000)),
    ("rf", RandomForestClassifier(n_estimators=200, random_state=42))
])

pipe.fit(X_train, y_train)
print(pipe.score(X_val, y_val))

0.843


In [62]:
test_dataset = text_dataset_from_directory("aclImdb/test")
print(f"Test samples: {len(test_dataset)}")

Test samples: 25000


In [63]:
pipe_unigrams = Pipeline([
    ("unigrams", UnigramTransformer(max_features=5000)),
    ("rf", RandomForestClassifier(n_estimators=200, random_state=42))
])
pipe_bigrams = Pipeline([
    ("bigrams", BigramTransformer(max_features=5000)),
    ("rf", RandomForestClassifier(n_estimators=200, random_state=42))
])
pipe_tfidf = Pipeline([
    ("tfidf", TfidfTransformer(max_features=5000)),
    ("rf", RandomForestClassifier(n_estimators=200, random_state=42))
])

models = {
    "Unigrams": pipe_unigrams,
    "Bigrams": pipe_bigrams,
    "TF-IDF": pipe_tfidf
}

for name, model in models.items():
    model.fit(dataset["text"], dataset["label"])
    score = model.score(test_dataset["text"], test_dataset["label"])
    print(f"{name} model test accuracy: {score:.4f}")

Unigrams model test accuracy: 0.8501
Bigrams model test accuracy: 0.8522
TF-IDF model test accuracy: 0.8472
